In [0]:
# Explorar la estructura del catálogo
display(dbutils.fs.ls("/Volumes/workspace/default/network_data/csv_extraidos/Network/Network/"))

In [0]:
from pyspark.sql.types import *

# ── AJUSTA ESTA RUTA ──────────────────────────────────────────────────────────
DATA_PATH = "/Volumes/workspace/default/network_data/csv_extraidos/Network/Network/*.csv"

schema = StructType([
    StructField("num",                         LongType(),   True),
    StructField("date",                        StringType(), True),
    StructField("time",                        StringType(), True),
    StructField("orig",                        StringType(), True),
    StructField("type",                        StringType(), True),
    StructField("i_f_name",                    StringType(), True),
    StructField("i_f_dir",                     StringType(), True),
    StructField("src",                         StringType(), True),
    StructField("dst",                         StringType(), True),
    StructField("proto",                       StringType(), True),
    StructField("appl_name",                   StringType(), True),
    StructField("proxy_src_ip",                StringType(), True),
    StructField("Modbus_Function_Code",        StringType(), True),
    StructField("Modbus_Function_Description", StringType(), True),
    StructField("Modbus_Transaction_ID",       StringType(), True),
    StructField("SCADA_Tag",                   StringType(), True),
    StructField("Modbus_Value",                StringType(), True),
    StructField("service",                     StringType(), True),
    StructField("s_port",                      StringType(), True),
    StructField("Tag",                         StringType(), True),
])

# ── LECTURA ───────────────────────────────────────────────────────────────────
df = spark.read.csv(DATA_PATH, header=True, schema=schema)

# ── VERIFICACIÓN ──────────────────────────────────────────────────────────────
print(f"Total registros: {df.count():,}")
display(df.limit(10))


In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. TIMESTAMP UNIFICADO Y SALTOS TEMPORALES
# ══════════════════════════════════════════════════════════════════════════════
from pyspark.sql import functions as F
# Crear timestamp combinando date + time
# La función de Spark reconocida es try_to_timestamp para no saltar excepciones, los pone a null quien no de con el formato.

df_ts = df.withColumn(
    "timestamp",
    F.coalesce(
        F.try_to_timestamp(
            F.concat_ws(" ", F.col("date"), F.col("time")),
            F.lit("dMMMyyyy H:mm:ss")       # 2Jan2016 2:08:50
        ),
        F.try_to_timestamp(
            F.concat_ws(" ", F.col("date"), F.col("time")),
            F.lit("d-MMM-yy H:mm:ss")       # 31-Dec-15 14:50:21
        )
    )
).withColumn(
    "timestamp_fmt",
    F.date_format(F.col("timestamp"), "dd/MM/yyyy h:mm:ss a")
)

# Vamos a ver cuantos son nulos.
total = df_ts.count()
nulos_ts = df_ts.filter(F.col("timestamp").isNull()).count()

print(f"Total registros:         {total:,}")
print(f"Timestamps nulos:        {nulos_ts:,}")
print(f"Porcentaje:              {nulos_ts/total*100:.2f}%")

# Ver qué valores de date y time no se pueden parsear
display(
    df_ts.filter(F.col("timestamp").isNull())
    .groupBy("date", "time")
    .agg(F.count("*").alias("count"))
    .orderBy("count", ascending=False)
    .limit(20)
)

In [0]:
# Eliminar registros donde timestamp es nulo
df_ts_clean = df_ts.filter(F.col("timestamp").isNotNull())

print(f"Registros después de borrar nulos: {df_ts_clean.count():,}")
display(df_ts_clean.limit(10))

In [0]:
DELTA_PATH = "/Volumes/workspace/default/network_data/delta/"

df_ts_clean \
    .repartition(200, "date") \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("date") \
    .save(DELTA_PATH)

print(f"Guardado en Delta Lake: {DELTA_PATH}")

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. INSPECCIÓN BÁSICA
# ══════════════════════════════════════════════════════════════════════════════


DELTA_PATH = "/Volumes/workspace/default/network_data/delta/"

df_analisis = spark.read.format("delta").load(DELTA_PATH)

print("\n=== ESTADÍSTICAS BÁSICAS ===")
display(df_analisis.describe())

In [0]:
from pyspark.sql import functions as F

print("=== NULOS POR COLUMNA ===")

null_counts = df_analisis.select([
    F.sum(
        F.when(
            F.col(c).isNull() |
            (
                (df_analisis.schema[c].dataType.simpleString() == "string") &
                (
                    (F.col(c) == "") |
                    (F.col(c) == "null") |
                    (F.col(c) == "NA") |
                    (F.col(c) == "-")
                )
            ),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df_analisis.columns
])

display(null_counts)